# Mock data setup for the Validating the "Critical Alert" System

In [15]:
import pandas as pd
import sqlite3
# create stimulated lap dataset
lab_data = {
"test_id": [701, 702, 703, 704, 705, 706, 707, 708],
"patient_id": ["P101", "P102", "P103", "P104", "P105", "P106", "P107", "P108"],
"potassium_level": [4.2, 2.8, 5.5, 6.2, 3.1, 7.5, 3.9, 2.5],
"ai_flag": ["NORMAL", "CRITICAL", "NORMAL", "NORMAL", "CRITICAL", "CRITICAL", "NORMAL", "NORMAL"]    
}

# 2.8 and 2.5 should be CRITICAL (AI missed one!)
# 6.2 and 7.5 should be CRITICAL (AI missed one!)
# 3.1 is NORMAL (AI hallucinated a flag!)

# add dataset to DataFrame

lab_dataFrame = pd.DataFrame(lab_data)

# connect to sqlite
connt = sqlite3.connect(":memory:")
# save the df to the temporary RAM
lab_dataFrame.to_sql("lab_results", connt, index = False, if_exists = ("replace"))

print('***** dataFrame for the Validating the Critical Alert has been created!!! *****')


***** dataFrame for the Validating the Critical Alert has been created!!! *****


# Finding the "False Negatives" (Missed Alerts)

In [19]:
# creating a function for query
def run_query(query):
    return pd.read_sql_query(query, connt)

# query -- for all data
all_data = "SELECT * FROM lab_results;"
# query --- finding the false negatives
false_negatives = """
SELECT * FROM lab_results WHERE potassium_level > 6 OR potassium_level < 3 AND ai_flag = 'NORMAL'
"""
print('************ all data table **********')
display(run_query(all_data))
print('**************** false negatives *************')
display(run_query(false_negatives))

************ all data table **********


,test_id,patient_id,potassium_level,ai_flag
0,701,P101,4.2,NORMAL
1,702,P102,2.8,CRITICAL
2,703,P103,5.5,NORMAL
3,704,P104,6.2,NORMAL
4,705,P105,3.1,CRITICAL
5,706,P106,7.5,CRITICAL
6,707,P107,3.9,NORMAL
7,708,P108,2.5,NORMAL


**************** false negatives *************


,test_id,patient_id,potassium_level,ai_flag
0,704,P104,6.2,NORMAL
1,706,P106,7.5,CRITICAL
2,708,P108,2.5,NORMAL


# Finding the "False Positives" (Hallucinated Alerts)

In [22]:
# query --- Finding the "False Positives" (Hallucinated Alerts)

false_postives = """
SELECT * FROM lab_results WHERE potassium_level BETWEEN 3.0 AND 6.0 AND ai_flag = 'CRITICAL'
"""
print('*********** false_postitives ********')
# rund the function for the query
display(run_query(false_postives))


*********** false_postitives ********


,test_id,patient_id,potassium_level,ai_flag
0,705,P105,3.1,CRITICAL
